In [1]:
# If your refactored file is at shared_lib/option.py
from options import (
    OptionAnalysisParams,
    analyze_options,
    format_option_results,
    pretty_view,
    round_numeric_columns,
)

symbols = [
    "UUUU",
    "LTBR",
    "BE",
    "SMR",
    "OKLO",
    "NNE",
    "CCCX",
    "WQTM",
    "QBTS",
    "QUBT",
    "IONQ",
    "RGTI",
    "QSI",
    "QS",
    "QNTM",
    "QMCO",
    "QBTQF",
    "ARQQ",
    "QTUM",
    "SLI",
    "DNUT",
    "KOPN",
    "KULR",
    "CLOV",
    "BYND",
    "NFE",
    "TLRY",
    "RR",
    "RKLB",
    "RBLX",
    "SERV",
    "BYRN",
    "BBAI",
    "APLD",
    "NVTS",
]

In [2]:
shares_owned = {"QUBT": 400}

contracts = analyze_options(
    OptionAnalysisParams(
        tickers=shares_owned.keys(),
        shares_owned=shares_owned,
        capital=None,  # not used for covered calls
        risk_free_rate=0.05,
        num_otm_strikes=6,
        num_atm_strikes=4,
    ),
)

2025-11-02 20:11:39.358 | INFO     | options:analyze_options:394 - Analyzing options with expirations up to 2025-12-31
2025-11-02 20:11:39.360 | INFO     | options:analyze_options:399 - Processing QUBT...


In [9]:
df = format_option_results(contracts)
df = df[df["Expiration"] < "2025-11-08"].drop(columns=["Expiration", "DTE", "Type"])
df_rounded = round_numeric_columns(df, money=2, percent=2, ratio=2, iv=1)
pretty = pretty_view(df_rounded, money=3, percent=2, ratio=4, iv=1)
pretty

,Ticker,Contracts,Capital Required,Strike,Spot,Strike vs Spot %,Total Premium,Premium / Capital %,Bid Premium %,Ask Premium %,Mid Premium %,Bid,Ask,Mid,BS Price,BS Ratio,IV,Delta,Theta
0,QUBT,4,$6600.000,$16.500,$16.710,-1.26%,$410.000,6.21%,6.06%,6.36%,6.21%,$1.000,$1.050,$1.020,$0.860,1.1900,108.0%,0.568766,-0.093999
1,QUBT,4,$6800.000,$17.000,$16.710,1.74%,$320.000,4.71%,4.41%,5.00%,4.71%,$0.750,$0.850,$0.800,$0.630,1.2600,109.0%,0.464556,-0.095672
2,QUBT,4,$7000.000,$17.500,$16.710,4.73%,$250.000,3.57%,3.43%,3.71%,3.57%,$0.600,$0.650,$0.620,$0.460,1.3500,110.9%,0.368823,-0.092275
3,QUBT,4,$7200.000,$18.000,$16.710,7.72%,$190.000,2.64%,2.50%,2.78%,2.64%,$0.450,$0.500,$0.480,$0.320,1.4700,111.5%,0.283019,-0.083118
4,QUBT,4,$7400.000,$18.500,$16.710,10.71%,$140.000,1.89%,1.62%,2.16%,1.89%,$0.300,$0.400,$0.350,$0.220,1.6100,111.3%,0.208912,-0.070392
5,QUBT,4,$7600.000,$19.000,$16.710,13.70%,$100.000,1.32%,1.05%,1.58%,1.32%,$0.200,$0.300,$0.250,$0.140,1.7900,110.5%,0.147506,-0.056050
6,QUBT,4,$7800.000,$19.500,$16.710,16.70%,$80.000,1.03%,0.77%,1.28%,1.03%,$0.150,$0.250,$0.200,$0.100,1.9700,114.5%,0.110433,-0.047430
7,QUBT,4,$8000.000,$20.000,$16.710,19.69%,$70.000,0.88%,0.75%,1.00%,0.87%,$0.150,$0.200,$0.180,$0.080,2.1000,121.5%,0.089264,-0.043068


In [11]:
puts_contracts = analyze_options(
    OptionAnalysisParams(
        tickers=symbols,
        # shares_owned={"QUBT": 400},
        capital=10000,  # Capital for cash-secured puts
        risk_free_rate=0.05,
        num_otm_strikes=6,
        num_atm_strikes=4,
    ),
)

2025-11-02 20:15:48.485 | INFO     | options:analyze_options:394 - Analyzing options with expirations up to 2025-12-31
2025-11-02 20:15:48.489 | INFO     | options:analyze_options:399 - Processing UUUU...
2025-11-02 20:15:49.428 | INFO     | options:analyze_options:399 - Processing LTBR...
2025-11-02 20:15:49.852 | INFO     | options:analyze_options:399 - Processing BE...
2025-11-02 20:15:50.796 | INFO     | options:analyze_options:399 - Processing SMR...
2025-11-02 20:15:51.685 | INFO     | options:analyze_options:399 - Processing OKLO...
2025-11-02 20:15:52.710 | INFO     | options:analyze_options:399 - Processing NNE...
2025-11-02 20:15:53.592 | INFO     | options:analyze_options:399 - Processing CCCX...
2025-11-02 20:15:54.065 | INFO     | options:analyze_options:399 - Processing WQTM...
2025-11-02 20:15:54.340 | WARNING  | options:get_option_chain:297 - No options expiring by 2025-12-31 for WQTM
2025-11-02 20:15:54.344 | INFO     | options:analyze_options:399 - Processing QBTS...


In [20]:
df_puts = format_option_results(puts_contracts)  # numeric, typed
df_puts_rounded = round_numeric_columns(df_puts, money=3, percent=2, ratio=4, iv=1)
df_puts_rounded = df_puts_rounded[df_puts_rounded["Bid"] > 0]  # filter out 0 bid options
df_puts_rounded = (
    df_puts_rounded[(df_puts_rounded["Expiration"] == "2025-11-07") & (df_puts_rounded["Strike vs Spot %"] < -5.0)]
    .sort_values(["Mid Premium %"], ascending=False)
    .drop(
        columns=["DTE", "Expiration", "Type", "Premium / Capital %"],
    )
)
pretty_puts = pretty_view(df_puts_rounded, money=3, percent=2, ratio=4, iv=1)[:50]
print("Expiring 2025-11-07 Cash-Secured Puts:")
pretty_puts

Expiring 2025-11-07 Cash-Secured Puts:


,Ticker,Contracts,Capital Required,Strike,Spot,Strike vs Spot %,Total Premium,Bid Premium %,Ask Premium %,Mid Premium %,Bid,Ask,Mid,BS Price,BS Ratio,IV,Delta,Theta
3,BYND,66,$9900.000,$1.500,$1.655,-9.37%,$1023.000,10.00%,10.67%,10.33%,$0.150,$0.160,$0.155,$0.115,1.3481,278.1%,-0.313766,-0.021274
28,SMR,2,$8400.000,$42.000,$44.870,-6.40%,$553.000,5.43%,7.74%,6.58%,$2.280,$3.250,$2.765,$2.052,1.3476,182.2%,-0.328248,-0.384352
18,QSI,50,$10000.000,$2.000,$2.150,-6.98%,$625.000,2.50%,10.00%,6.25%,$0.050,$0.200,$0.125,$0.091,1.3680,179.7%,-0.315104,-0.017856
20,NVTS,8,$10000.000,$12.500,$13.460,-7.13%,$600.000,5.60%,6.40%,6.00%,$0.700,$0.800,$0.750,$0.548,1.3698,176.4%,-0.309929,-0.108934
50,QBTS,2,$7000.000,$35.000,$37.060,-5.56%,$409.000,5.40%,6.29%,5.84%,$1.890,$2.200,$2.045,$1.520,1.3452,161.0%,-0.334767,-0.282524
40,SMR,2,$8300.000,$41.500,$44.870,-7.51%,$475.000,3.01%,8.43%,5.72%,$1.250,$3.500,$2.375,$1.717,1.3832,173.6%,-0.300335,-0.352566
39,SMR,2,$8500.000,$42.500,$44.870,-5.28%,$480.000,3.65%,7.65%,5.65%,$1.550,$3.250,$2.400,$1.786,1.3436,155.0%,-0.337619,-0.330202
37,UUUU,5,$9500.000,$19.000,$20.510,-7.36%,$487.500,5.00%,5.26%,5.13%,$0.950,$1.000,$0.975,$0.699,1.3957,160.5%,-0.293803,-0.147470
47,SMR,2,$8200.000,$41.000,$44.870,-8.62%,$421.000,4.17%,6.10%,5.13%,$1.710,$2.500,$2.105,$1.483,1.4189,170.0%,-0.274639,-0.330989
60,QBTS,2,$6900.000,$34.500,$37.060,-6.91%,$348.000,4.38%,5.71%,5.04%,$1.510,$1.970,$1.740,$1.253,1.3889,155.4%,-0.299870,-0.260233
